In [1]:
"""
Self-Pruning Neural Network — Tredence AI Engineering Case Study A feed-forward network that learns to prune itself during training using learnable gate parameters and L1 sparsity regularization.

Author  : Jesvanth S
Dataset : CIFAR-10
"""

import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


# PART 1 ─ PrunableLinear Layer

class PrunableLinear(nn.Module):
    """
    A custom linear layer augmented with learnable gate_scores.

    For each weight w_ij, a corresponding gate_score s_ij is maintained.
    During the forward pass:
        gate   = sigmoid(gate_score)          ∈ (0, 1)
        pruned = weight ⊙ gate               element-wise
        out    = x @ pruned.T + bias

    When a gate collapses to ≈0, the weight is effectively removed.
    Gradients flow through both `weight` and `gate_scores` automatically
    because all operations are differentiable PyTorch ops.
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        # Standard weight & bias (same init as nn.Linear)
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))

        # Learnable gate scores — same shape as weight
        # Initialised near 0 so sigmoid(0) ≈ 0.5 → gates start half-open
        self.gate_scores = nn.Parameter(torch.zeros(out_features, in_features))

        # Kaiming uniform init for weight (standard practice)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        # Small noise on gate_scores for symmetry breaking
        nn.init.normal_(self.gate_scores, mean=0.0, std=0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Step 1 — turn raw scores into gates ∈ (0, 1)
        gates = torch.sigmoid(self.gate_scores)

        # Step 2 — element-wise multiplication prunes weak weights
        pruned_weights = self.weight * gates

        # Step 3 — standard linear operation (F.linear = x @ W.T + b)
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self) -> torch.Tensor:
        """Return current gate values (detached, for analysis)."""
        return torch.sigmoid(self.gate_scores).detach()

    def sparsity(self, threshold: float = 1e-2) -> float:
        """Fraction of gates below threshold (i.e. pruned)."""
        gates = self.get_gates()
        return (gates < threshold).float().mean().item()

    def extra_repr(self) -> str:
        return f"in={self.in_features}, out={self.out_features}"


# Network Definition

class SelfPruningNet(nn.Module):
    """
    Feed-forward network for CIFAR-10 (3×32×32 → 10 classes).
    All linear layers are PrunableLinear so every weight has a gate.
    """

    def __init__(self):
        super().__init__()
        # CIFAR-10 images: 3 channels × 32 × 32 = 3072 input features
        self.fc1 = PrunableLinear(3072, 1024)
        self.fc2 = PrunableLinear(1024, 512)
        self.fc3 = PrunableLinear(512,  256)
        self.fc4 = PrunableLinear(256,  10)

        self.bn1 = nn.BatchNorm1d(1024)
        self.bn2 = nn.BatchNorm1d(512)
        self.bn3 = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)          # flatten spatial dims
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        return x

    def prunable_layers(self):
        """Yield all PrunableLinear layers."""
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                yield m

    def sparsity_loss(self) -> torch.Tensor:
        """
        L1 norm of ALL gate values across every PrunableLinear layer.

        L1 is chosen because:
        - It penalises each gate proportional to its magnitude (not square).
        - This creates a constant gradient pressure toward zero regardless of
          the current gate value — unlike L2 whose gradient vanishes near 0.
        - Result: many gates are pushed all the way to 0 (true sparsity).
        """
        total = torch.tensor(0.0, requires_grad=True)
        for layer in self.prunable_layers():
            gates = torch.sigmoid(layer.gate_scores)   # keep in compute graph
            total = total + gates.abs().sum()
        return total

    def overall_sparsity(self, threshold: float = 1e-2) -> float:
        """Global fraction of pruned weights."""
        pruned = total = 0
        for layer in self.prunable_layers():
            g = layer.get_gates()
            pruned += (g < threshold).sum().item()
            total  += g.numel()
        return pruned / total if total > 0 else 0.0

    def all_gate_values(self) -> np.ndarray:
        """Collect all gate values into a single numpy array."""
        all_gates = []
        for layer in self.prunable_layers():
            all_gates.append(layer.get_gates().cpu().numpy().ravel())
        return np.concatenate(all_gates)


# Data Loading

def get_cifar10_loaders(batch_size: int = 128, data_dir: str = "./data"):
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2470, 0.2435, 0.2616)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2470, 0.2435, 0.2616)),
    ])

    train_set = torchvision.datasets.CIFAR10(
        root=data_dir, train=True,  download=True, transform=transform_train)
    test_set  = torchvision.datasets.CIFAR10(
        root=data_dir, train=False, download=True, transform=transform_test)

    train_loader = DataLoader(train_set, batch_size=batch_size,
                              shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=256,
                              shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, test_loader


# PART 3 ─ Training Loop

def train_one_epoch(model, loader, optimizer, lam, device):
    model.train()
    running_loss = running_cls_loss = running_spar_loss = 0.0
    correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        logits = model(images)

        # Classification loss (cross-entropy)
        cls_loss  = F.cross_entropy(logits, labels)

        # Sparsity regularisation — L1 on all gate values
        spar_loss = model.sparsity_loss()

        # Total loss
        loss = cls_loss + lam * spar_loss

        loss.backward()
        optimizer.step()

        running_loss      += loss.item()
        running_cls_loss  += cls_loss.item()
        running_spar_loss += spar_loss.item()

        preds    = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    n = len(loader)
    return {
        "loss"      : running_loss      / n,
        "cls_loss"  : running_cls_loss  / n,
        "spar_loss" : running_spar_loss / n,
        "accuracy"  : correct / total * 100,
    }


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total * 100

# PART 3 ─ Evaluation Helpers

def run_experiment(lam: float, device, train_loader, test_loader,
                   epochs: int = 30, lr: float = 1e-3) -> dict:
    """Train one model with a given λ and return results."""
    print(f"\n{'='*60}")
    print(f"  λ = {lam}   |  epochs = {epochs}")
    print(f"{'='*60}")

    model     = SelfPruningNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = []

    for epoch in range(1, epochs + 1):
        stats = train_one_epoch(model, train_loader, optimizer, lam, device)
        scheduler.step()

        if epoch % 5 == 0 or epoch == epochs:
            sparsity = model.overall_sparsity() * 100
            test_acc = evaluate(model, test_loader, device)
            print(f"  Epoch {epoch:3d}  |  "
                  f"Loss={stats['loss']:.4f}  "
                  f"CLS={stats['cls_loss']:.4f}  "
                  f"SPAR={stats['spar_loss']:.1f}  |  "
                  f"TrainAcc={stats['accuracy']:.1f}%  "
                  f"TestAcc={test_acc:.1f}%  "
                  f"Sparsity={sparsity:.1f}%")

        history.append(stats)

    final_test_acc = evaluate(model, test_loader, device)
    final_sparsity = model.overall_sparsity() * 100
    gate_values    = model.all_gate_values()

    print(f"\n  ✓ Final Test Accuracy : {final_test_acc:.2f}%")
    print(f"  ✓ Sparsity Level      : {final_sparsity:.2f}%")

    return {
        "lambda"      : lam,
        "test_acc"    : final_test_acc,
        "sparsity"    : final_sparsity,
        "gate_values" : gate_values,
        "history"     : history,
        "model"       : model,
    }

# Plotting

def plot_gate_distribution(results: list, save_path: str = "gate_distributions.png"):
    """
    Plot gate value distributions for all λ values.
    A successful run shows a large spike near 0 (pruned) and
    a cluster of larger values (active weights).
    """
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), sharey=False)
    if n == 1:
        axes = [axes]

    colors = ["#e63946", "#2a9d8f", "#e9c46a"]

    for ax, res, color in zip(axes, results, colors):
        gates = res["gate_values"]
        ax.hist(gates, bins=80, color=color, alpha=0.85, edgecolor="none")
        ax.set_title(
            f"λ = {res['lambda']}\n"
            f"Test Acc: {res['test_acc']:.1f}%  |  Sparsity: {res['sparsity']:.1f}%",
            fontsize=11, fontweight="bold"
        )
        ax.set_xlabel("Gate Value", fontsize=10)
        ax.set_ylabel("Count",      fontsize=10)
        ax.axvline(x=0.01, color="black", linestyle="--",
                   linewidth=1.2, label="Prune threshold (0.01)")
        ax.legend(fontsize=8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle("Gate Value Distributions — Self-Pruning Neural Network",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  Plot saved → {save_path}")


def plot_training_curves(results: list, save_path: str = "training_curves.png"):
    """Plot classification loss curves across epochs for all λ values."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ["#e63946", "#2a9d8f", "#e9c46a"]

    for res, color in zip(results, colors):
        epochs  = range(1, len(res["history"]) + 1)
        cls_losses = [h["cls_loss"]  for h in res["history"]]
        train_accs = [h["accuracy"]  for h in res["history"]]

        axes[0].plot(epochs, cls_losses, label=f"λ={res['lambda']}", color=color, lw=2)
        axes[1].plot(epochs, train_accs,  label=f"λ={res['lambda']}", color=color, lw=2)

    for ax, title, ylabel in zip(
        axes,
        ["Classification Loss vs Epochs", "Train Accuracy vs Epochs"],
        ["Cross-Entropy Loss", "Accuracy (%)"]
    ):
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Plot saved → {save_path}")

# Main
def main():
    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n  Device : {device}")

    # Reproducibility
    torch.manual_seed(42)
    np.random.seed(42)

    # Data
    train_loader, test_loader = get_cifar10_loaders(batch_size=128)

    # Lambda values to compare: low / medium / high
    lambdas = [1e-5, 1e-4, 1e-3]

    results = []
    for lam in lambdas:
        res = run_experiment(
            lam=lam,
            device=device,
            train_loader=train_loader,
            test_loader=test_loader,
            epochs=30,
            lr=1e-3,
        )
        results.append(res)

    # ── Results Table ──────────────────────────────────────
    print("\n")
    print("=" * 55)
    print(f"  {'Lambda':<12} {'Test Accuracy':>15} {'Sparsity (%)':>15}")
    print("=" * 55)
    for r in results:
        print(f"  {r['lambda']:<12} {r['test_acc']:>14.2f}%  {r['sparsity']:>13.2f}%")
    print("=" * 55)

    # ── Plots ─────────────────────────────────────────────
    plot_gate_distribution(results, save_path="gate_distributions.png")
    plot_training_curves(results,   save_path="training_curves.png")

    print("\n  Done. Check gate_distributions.png and training_curves.png\n")


if __name__ == "__main__":
    main()


  Device : cpu


100%|██████████| 170M/170M [00:04<00:00, 38.9MB/s]



  λ = 1e-05   |  epochs = 30


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Epoch   5  |  Loss=20.3380  CLS=1.5570  SPAR=1878103.4  |  TrainAcc=43.6%  TestAcc=48.0%  Sparsity=0.0%
  Epoch  10  |  Loss=20.2569  CLS=1.4758  SPAR=1878105.4  |  TrainAcc=46.7%  TestAcc=51.0%  Sparsity=0.0%
  Epoch  15  |  Loss=20.1902  CLS=1.4091  SPAR=1878115.0  |  TrainAcc=49.2%  TestAcc=54.0%  Sparsity=0.0%
  Epoch  20  |  Loss=20.1231  CLS=1.3418  SPAR=1878130.0  |  TrainAcc=51.5%  TestAcc=56.6%  Sparsity=0.0%
  Epoch  25  |  Loss=20.0690  CLS=1.2876  SPAR=1878145.3  |  TrainAcc=53.6%  TestAcc=58.1%  Sparsity=0.0%
  Epoch  30  |  Loss=20.0452  CLS=1.2636  SPAR=1878154.4  |  TrainAcc=54.5%  TestAcc=58.1%  Sparsity=0.0%

  ✓ Final Test Accuracy : 58.12%
  ✓ Sparsity Level      : 0.00%

  λ = 0.0001   |  epochs = 30
  Epoch   5  |  Loss=168.4478  CLS=1.5543  SPAR=1668935.0  |  TrainAcc=44.1%  TestAcc=48.6%  Sparsity=0.0%
  Epoch  10  |  Loss=168.3675  CLS=1.4757  SPAR=1668917.7  |  TrainAcc=46.6%  TestAcc=51.5%  Sparsity=0.0%
  Epoch  15  |  Loss=168.3035  CLS=1.4113  SPAR=16689